# 01 — TF-IDF Rating-Sentiment Baseline

This notebook trains the classical TF-IDF baseline on the same 999,999 Amazon Beauty reviews used to train DistilBERT. Both models then use the same saved validation and final-test review IDs. This makes the comparison reflect the modeling method rather than a difference in training-data size.

## Objectives

- verify the corrected UTC `v2` reference and `1m_v3` experiment manifests;
- verify 333,333 training reviews per rating-derived class;
- select TF-IDF + Logistic Regression settings using validation rows only;
- evaluate the new model on the exact product, time, and `Conditioners` tests used by DistilBERT;
- retrain versioned 300k-reference and 1M TF-IDF models without overwriting history;
- compare training sizes only on shared corrected test IDs and save both reports.

# Базовая модель тональности TF-IDF

Этот ноутбук обучает базовую модель TF-IDF на тех же 999 999 отзывах Amazon Beauty, которые использовались для DistilBERT. Затем обе модели проверяются на одинаковых сохранённых `review_id`. Благодаря этому сравниваются методы моделей, а не разное количество обучающих данных.

## Цели

- проверить исправленные UTC-разбиения `v2` для reference и `1m_v3` для основного эксперимента;
- проверить наличие 333 333 обучающих отзывов для каждого класса по звёздам;
- выбрать настройки TF-IDF + Logistic Regression только по данным для настройки модели;
- проверить новую модель на тех же товарах, периоде и `Conditioners`, что и DistilBERT;
- заново обучить версионированные TF-IDF-модели на 300 тысячах и одном миллионе строк, не перезаписывая историю;
- сравнить размеры обучения только на общих исправленных test ID и сохранить оба отчёта.

## 1. Inputs, outputs, and assumptions

**Inputs:** the rebuilt Beauty reviews, one-row-per-product catalog, and corrected UTC manifests `beauty_rating_sentiment_split_v2` and `beauty_rating_sentiment_split_1m_v3`. **Outputs:** two new local TF-IDF model directories, predictions, and JSON reports.

The target is a **weak label derived from stars**, not manually annotated human sentiment: ratings 1–2 are `negative`, rating 3 is `neutral`, and ratings 4–5 are `positive`. Text can disagree with its rating, so the model cannot be treated as the final truth about every sentence.

## Входы, результаты и допущения

**Входы:** пересобранные отзывы Beauty, каталог с одной строкой на товар и исправленные UTC-разбиения `beauty_rating_sentiment_split_v2` и `beauty_rating_sentiment_split_1m_v3`. **Результаты:** две новые локальные папки TF-IDF, предсказания и JSON-отчёты.

Цель обучения — **приблизительная метка, полученная из звёзд**, а не тональность, вручную проверенная человеком: оценки 1–2 дают `negative`, оценка 3 — `neutral`, оценки 4–5 — `positive`. Текст может не совпадать со звёздами, поэтому модель нельзя считать окончательной истиной о каждом предложении.

In [ ]:
# Standard library / Стандартная библиотека
import json
import platform
import sys
import time
from pathlib import Path

# Third-party packages / Сторонние библиотеки
import duckdb
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import sklearn

# Locate the repository before importing reusable project code.
# Находим репозиторий до импорта переиспользуемого кода проекта.
PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "PLAN.md").is_file() and (candidate / "src").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Local project modules / Локальные модули проекта
from src.ingestion.dataset_manifest import load_dataset_manifest
from src.ml.sentiment import (
    SENTIMENT_LABELS,
    SentimentSplitConfig,
    TfidfBaselineConfig,
    audit_sentiment_sample,
    balanced_evaluation_view,
    evaluate_predictions,
    load_sentiment_split_reviews,
    predict_sentiment,
    save_sentiment_experiment,
    select_tfidf_baseline,
)

In [ ]:
# Keep corrected reference and 1M experiments under new immutable versions.
# Храним исправленные reference- и 1M-эксперименты под новыми неизменяемыми версиями.
DATASET_VERSION = "amazon_reviews_2023_beauty_2021_2023_v1"
MANIFEST_PATH = (
    PROJECT_ROOT / "config/datasets" / f"{DATASET_VERSION}.json"
)
DATASET_DIRECTORY = PROJECT_ROOT / "data/processed" / DATASET_VERSION
REFERENCE_SPLIT_CONFIG = SentimentSplitConfig(
    split_version="beauty_rating_sentiment_split_v2",
)
SPLIT_CONFIG = SentimentSplitConfig(
    split_version="beauty_rating_sentiment_split_1m_v3",
    train_per_class=333_333,
)
REFERENCE_MODEL_CONFIG = TfidfBaselineConfig(
    model_version="beauty_tfidf_rating_sentiment_v2",
)
MODEL_CONFIG = TfidfBaselineConfig(
    model_version="beauty_tfidf_rating_sentiment_1m_v3",
)
REFERENCE_MODEL_DIRECTORY = (
    PROJECT_ROOT / "models" / REFERENCE_MODEL_CONFIG.model_version
)
MODEL_DIRECTORY = PROJECT_ROOT / "models/sentiment-baseline"
REFERENCE_SPLIT_MANIFEST_PATH = (
    DATASET_DIRECTORY
    / "ml_splits"
    / f"{REFERENCE_SPLIT_CONFIG.split_version}.parquet"
)
SPLIT_MANIFEST_PATH = (
    DATASET_DIRECTORY / "ml_splits" / f"{SPLIT_CONFIG.split_version}.parquet"
)
PREDICTIONS_PATH = (
    DATASET_DIRECTORY
    / "model_predictions"
    / f"{MODEL_CONFIG.model_version}.parquet"
)
REFERENCE_PREDICTIONS_PATH = (
    DATASET_DIRECTORY
    / "model_predictions"
    / f"{REFERENCE_MODEL_CONFIG.model_version}.parquet"
)
REPORT_PATH = (
    PROJECT_ROOT
    / "reports/model_evaluation"
    / f"{MODEL_CONFIG.model_version}.json"
)
REFERENCE_REPORT_PATH = (
    PROJECT_ROOT
    / "reports/model_evaluation"
    / f"{REFERENCE_MODEL_CONFIG.model_version}.json"
)

display(
    pd.Series(
        {
            "dataset_version": DATASET_VERSION,
            "reference_split_version": REFERENCE_SPLIT_CONFIG.split_version,
            "one_million_split_version": SPLIT_CONFIG.split_version,
            "training_reviews": SPLIT_CONFIG.train_per_class * 3,
            "reference_model_version": REFERENCE_MODEL_CONFIG.model_version,
            "one_million_model_version": MODEL_CONFIG.model_version,
            "temporal_cutoff": SPLIT_CONFIG.temporal_cutoff,
            "random_state": SPLIT_CONFIG.random_state,
        },
        name="value",
    ).to_frame()
)

## 2. Environment and source-data checks

The notebook reads file paths from the registered dataset description and validates them before expensive work. The RAPIDS kernel supplies the project environment, while this classical baseline itself trains on the CPU through scikit-learn.

## Окружение и проверка исходных данных

Ноутбук получает пути из зарегистрированного описания датасета и проверяет файлы до долгих вычислений. Ядро RAPIDS предоставляет окружение проекта, но сама классическая модель обучается на процессоре через scikit-learn.

In [ ]:
manifest = load_dataset_manifest(MANIFEST_PATH)
reviews_file = manifest.file_by_role("canonical_reviews")
catalog_file = manifest.file_by_role("product_catalog")
REVIEWS_PATH = PROJECT_ROOT / reviews_file.path
CATALOG_PATH = PROJECT_ROOT / catalog_file.path

assert manifest.dataset_version == DATASET_VERSION
assert REVIEWS_PATH.is_file()
assert CATALOG_PATH.is_file()
assert REVIEWS_PATH.stat().st_size == reviews_file.size_bytes
assert CATALOG_PATH.stat().st_size == catalog_file.size_bytes

environment_summary = pd.Series(
    {
        "python": platform.python_version(),
        "python_executable": sys.executable,
        "scikit_learn": sklearn.__version__,
        "duckdb": duckdb.__version__,
        "canonical_review_rows": reviews_file.record_count,
        "catalog_product_rows": catalog_file.record_count,
    },
    name="value",
).to_frame()
display(environment_summary)

## 3. Exact label population

Before sampling, we count every canonical review. This table describes the real Beauty population and shows why training is balanced: positive ratings dominate, while a classifier trained naively could obtain misleading accuracy by favoring the majority class.

## Точное распределение меток

До создания выборки считаем все канонические отзывы. Таблица описывает реальную совокупность Beauty и показывает, зачем балансировать обучение: положительные рейтинги преобладают, поэтому наивная модель могла бы получить обманчиво высокую accuracy, предпочитая самый частый класс.

In [ ]:
connection = duckdb.connect()
try:
    population_distribution = connection.execute(
        """
        SELECT
            CASE
                WHEN rating IN (1, 2) THEN 'negative'
                WHEN rating = 3 THEN 'neutral'
                ELSE 'positive'
            END AS sentiment_label,
            count(*) AS review_count,
            round(100.0 * count(*) / sum(count(*)) OVER (), 2) AS share_pct
        FROM read_parquet(?)
        GROUP BY sentiment_label
        ORDER BY sentiment_label
        """,
        [str(REVIEWS_PATH)],
    ).fetchdf()
finally:
    connection.close()

assert population_distribution["review_count"].sum() == reviews_file.record_count
display(population_distribution)

## 4. Use the same products and dates as DistilBERT

The saved split groups reviews by `parent_asin`. Historical reviews from 80% of stable product groups supply training, 10% supply validation, and the remaining products supply final testing. Reviews from 2023 form a later-review test only when the product is represented in training. `test_conditioners` contains held-out Conditioner parent products; it does not exclude the whole Conditioner niche from training.

The split retains one normalized copy of repeated text and prevents the same normalized text from crossing group boundaries. Training and validation contain equal numbers of the three labels. Final tests preserve the real label distribution. User overlap is measured but not prohibited because the main business question is whether the model works for products outside training.

## Используем те же товары и даты, что и DistilBERT

Сохранённое разбиение группирует отзывы по `parent_asin`. Исторические отзывы из 80% стабильных групп товаров используются для обучения, 10% — для выбора настроек, а оставшиеся товары — для итоговой проверки. Отзывы 2023 года входят в проверку на более поздних данных, только если товар представлен при обучении. `Conditioners` — отдельная проверка товаров, отзывы о которых не использовались при обучении.

Разбиение оставляет одну нормализованную копию повторяющегося текста и не позволяет одинаковому тексту попасть в разные группы. В обучении и данных для выбора настроек три метки представлены поровну. Итоговые проверки сохраняют реальное распределение оценок. Пересечение покупателей измеряется, но не запрещается: главный бизнес-вопрос — работает ли модель для товаров вне обучения.

In [ ]:
# Load the corrected manifests and restore text only for the active 1M run.
# Загружаем исправленные manifests, а тексты присоединяем только для активного 1M-запуска.
reference_manifest = pd.read_parquet(REFERENCE_SPLIT_MANIFEST_PATH)
sentiment_sample = load_sentiment_split_reviews(
    REVIEWS_PATH,
    SPLIT_MANIFEST_PATH,
)
split_distribution = (
    sentiment_sample.groupby(["split_name", "sentiment_label"])
    .size()
    .rename("review_count")
    .reset_index()
)

training_counts = (
    sentiment_sample.loc[sentiment_sample["split_name"].eq("train")]
    ["sentiment_label"]
    .value_counts()
)
assert len(reference_manifest) == 569_021
assert len(sentiment_sample) == 1_269_020
assert str(reference_manifest["review_timestamp"].dt.tz) == "UTC"
assert str(sentiment_sample["review_timestamp"].dt.tz) == "UTC"
assert not reference_manifest["review_id"].duplicated().any()
assert training_counts.to_dict() == {
    "negative": 333_333,
    "neutral": 333_333,
    "positive": 333_333,
}
reference_distribution = (
    reference_manifest.groupby(["split_name", "sentiment_label"])
    .size()
    .rename("review_count")
    .reset_index()
)
display(reference_distribution.assign(experiment="300k reference"))
display(split_distribution.assign(experiment="1M"))

In [ ]:
reference_audit = audit_sentiment_sample(reference_manifest)
reference_audit_table = pd.DataFrame(reference_audit["splits"])
split_audit = audit_sentiment_sample(sentiment_sample)
split_audit_table = pd.DataFrame(split_audit["splits"])

assert reference_audit["review_id_is_unique"]
assert reference_audit["text_fingerprint_is_unique"]
assert split_audit["review_id_is_unique"]
assert split_audit["text_fingerprint_is_unique"]
assert (
    (split_audit_table["train_text_overlap"] == 0)
    | (split_audit_table["split_name"] == "train")
).all()
assert (
    split_audit_table.loc[
        split_audit_table["split_name"].isin(
            ["validation", "test_product", "test_conditioners"]
        ),
        "train_product_overlap",
    ]
    == 0
).all()
cutoff = pd.Timestamp(SPLIT_CONFIG.temporal_cutoff, tz="UTC")
for checked_manifest in (reference_manifest, sentiment_sample):
    historical = checked_manifest["split_name"].ne("test_temporal_seen")
    assert (checked_manifest.loc[historical, "review_timestamp"] < cutoff).all()
    assert (checked_manifest.loc[~historical, "review_timestamp"] >= cutoff).all()
display(reference_audit_table.assign(experiment="300k reference"))
display(split_audit_table.assign(experiment="1M"))

## 5. Train TF-IDF on 999,999 reviews

TF-IDF converts words and neighboring word pairs into weighted numerical signs. Logistic Regression learns which signs are associated with the three star-derived classes. Two regularization settings are compared using only validation macro F1; no final-test result influences the choice.

This is CPU training and can take noticeably longer than the historical 300k run. The elapsed time is recorded so future category runs can be planned realistically.

## Обучаем TF-IDF на 999 999 отзывах

TF-IDF превращает слова и соседние пары слов во взвешенные числовые признаки. Затем Logistic Regression учится связывать эти признаки с тремя классами по звёздам. Два значения регуляризации сравниваются только по macro F1 на данных для выбора настроек; результаты итоговых проверок не влияют на выбор.

Обучение выполняется на процессоре и может занять заметно больше времени, чем старый запуск на 300 тысячах строк. Время записывается, чтобы реалистично планировать будущие товарные категории.

In [ ]:
train_reviews = sentiment_sample.loc[
    sentiment_sample["split_name"] == "train"
].copy()
validation_reviews = sentiment_sample.loc[
    sentiment_sample["split_name"] == "validation"
].copy()

# Time both candidate settings because this is the expensive CPU operation.
# Измеряем оба варианта настроек, потому что это основная долгая операция на CPU.
training_started = time.perf_counter()
vectorizer, classifier, validation_selection = select_tfidf_baseline(
    train_reviews,
    validation_reviews,
    config=MODEL_CONFIG,
)
training_duration_seconds = time.perf_counter() - training_started

assert len(train_reviews) == 999_999
assert len(vectorizer.vocabulary_) <= MODEL_CONFIG.max_features
assert tuple(classifier.classes_) == SENTIMENT_LABELS
display(validation_selection)
display(
    pd.Series(
        {
            "training_reviews": len(train_reviews),
            "training_duration_seconds": training_duration_seconds,
            "selected_regularization_c": float(classifier.C),
            "vocabulary_size": len(vectorizer.vocabulary_),
        },
        name="value",
    ).to_frame()
)

## 6. Evaluate both models on identical final tests

`test_product` checks Beauty products whose reviews were excluded from training. `test_temporal_seen` checks later 2023 language for products represented in training. `test_conditioners` checks the first candidate Amazon niche without using its product reviews for training. Natural views retain the real label frequency; balanced views give each class equal weight and expose Neutral weakness.

The shown 95% intervals use percentile bootstrap resampling of whole `parent_asin` clusters, so reviews from the same product move together. `model_score` remains a model estimate; Error Analysis checks separately whether it matches the observed correct-answer share.

## Проверяем обе модели на одинаковых итоговых данных

`test_product` проверяет товары Beauty, отзывы о которых не использовались при обучении. `test_temporal_seen` проверяет более поздние тексты 2023 года для товаров из обучения. `test_conditioners` проверяет первую выбранную нишу Amazon без использования отзывов её проверочных товаров при обучении. Естественные срезы сохраняют настоящую частоту классов, а сбалансированные дают классам равный вес и лучше показывают слабость Neutral.

Показанные 95%-е интервалы используют percentile bootstrap целых кластеров `parent_asin`: отзывы одного товара всегда перемещаются вместе. `model_score` остаётся оценкой модели; отдельный Error Analysis проверяет, соответствует ли она реальной доле правильных ответов.

In [ ]:
evaluation_reviews = sentiment_sample.loc[
    sentiment_sample["split_name"] != "train"
].copy()
evaluation_predictions = predict_sentiment(
    evaluation_reviews,
    vectorizer=vectorizer,
    classifier=classifier,
)

evaluation_views = {
    "validation_balanced": evaluation_predictions.loc[
        evaluation_predictions["split_name"] == "validation"
    ]
}
for split_name in (
    "test_product",
    "test_temporal_seen",
    "test_conditioners",
):
    natural_view = evaluation_predictions.loc[
        evaluation_predictions["split_name"] == split_name
    ]
    evaluation_views[f"{split_name}_natural"] = natural_view
    evaluation_views[f"{split_name}_balanced"] = (
        balanced_evaluation_view(
            natural_view,
            max_per_class=20_000,
            random_state=MODEL_CONFIG.random_state,
        )
    )

metrics = {
    "training": {
        "review_count": int(len(train_reviews)),
        "duration_seconds": float(training_duration_seconds),
        "selected_regularization_c": float(classifier.C),
        "vocabulary_size": int(len(vectorizer.vocabulary_)),
    }
}
metric_rows = []
for view_name, view in evaluation_views.items():
    view_metrics = evaluate_predictions(
        view,
        bootstrap_rounds=200,
        random_state=MODEL_CONFIG.random_state,
    )
    metrics[view_name] = view_metrics
    metric_rows.append(
        {
            "evaluation_view": view_name,
            "reviews": view_metrics["review_count"],
            "accuracy": view_metrics["accuracy"],
            "macro_f1": view_metrics["macro_f1"],
            "weighted_f1": view_metrics["weighted_f1"],
            "neutral_f1": view_metrics["per_class"]["neutral"][
                "f1-score"
            ],
            "macro_f1_ci_low": view_metrics[
                "confidence_intervals_95"
            ]["macro_f1"][0],
            "macro_f1_ci_high": view_metrics[
                "confidence_intervals_95"
            ]["macro_f1"][1],
        }
    )

metrics_table = pd.DataFrame(metric_rows)
display(metrics_table.round(4))

In [ ]:
# Compare the broad product test with the first candidate niche.
# Сравниваем общую проверку по товарам с первой выбранной нишей.
figure, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for axis, split_name, title in (
    (axes[0], "test_product", "Beauty products excluded from training"),
    (axes[1], "test_conditioners", "Held-out Conditioner parent products"),
):
    matrix = metrics[f"{split_name}_natural"]["confusion_matrix"]
    sns.heatmap(
        matrix,
        annot=True,
        fmt=",",
        cmap="Blues",
        xticklabels=SENTIMENT_LABELS,
        yticklabels=SENTIMENT_LABELS,
        ax=axis,
    )
    axis.set_title(title)
    axis.set_xlabel("Model answer")
    axis.set_ylabel("Label created from star rating")
figure.suptitle("TF-IDF baseline confusion matrices")
figure.tight_layout()
plt.show()

## 7. Retrain the corrected 300k reference and compare shared tests

The old 300k report cannot be reused: its manifest was created with a session-local interpretation of the timezone-free cutoff. We therefore retrain a new immutable reference on `beauty_rating_sentiment_split_v2`, evaluate it with product-cluster bootstrap, and save it under a new model ID.

The corrected 300k and 1M manifests are not assumed to have identical evaluation membership. Training-size comparisons join predictions by `review_id` and recompute both metrics on the exact shared rows; coverage is reported explicitly.

## Переобучаем исправленную 300k-reference и сравниваем общие тесты

Старый 300k-отчёт нельзя переиспользовать: его manifest создавался с зависимой от сессии трактовкой временной границы без timezone. Поэтому мы переобучаем новую неизменяемую reference-модель на `beauty_rating_sentiment_split_v2`, оцениваем её cluster bootstrap по товарам и сохраняем под новым ID.

Мы не считаем evaluation membership исправленных 300k- и 1M-разбиений одинаковым. Для сравнения размера обучения предсказания соединяются по `review_id`, обе метрики пересчитываются на строго общих строках, а coverage показывается явно.

In [ ]:
# Restore corrected reference texts, train, and evaluate under a new ID.
# Восстанавливаем тексты исправленной reference, обучаем и оцениваем новый ID.
reference_sample = load_sentiment_split_reviews(
    REVIEWS_PATH,
    REFERENCE_SPLIT_MANIFEST_PATH,
)
reference_train = reference_sample.loc[
    reference_sample["split_name"].eq("train")
].copy()
reference_validation = reference_sample.loc[
    reference_sample["split_name"].eq("validation")
].copy()
reference_training_started = time.perf_counter()
reference_vectorizer, reference_classifier, reference_selection = (
    select_tfidf_baseline(
        reference_train,
        reference_validation,
        config=REFERENCE_MODEL_CONFIG,
    )
)
reference_training_seconds = time.perf_counter() - reference_training_started
reference_evaluation = reference_sample.loc[
    reference_sample["split_name"].ne("train")
].copy()
reference_predictions = predict_sentiment(
    reference_evaluation,
    vectorizer=reference_vectorizer,
    classifier=reference_classifier,
)
reference_views = {
    "validation_balanced": reference_predictions.loc[
        reference_predictions["split_name"].eq("validation")
    ]
}
for split_name in (
    "test_product",
    "test_temporal_seen",
    "test_conditioners",
):
    natural_view = reference_predictions.loc[
        reference_predictions["split_name"].eq(split_name)
    ]
    reference_views[f"{split_name}_natural"] = natural_view
    reference_views[f"{split_name}_balanced"] = balanced_evaluation_view(
        natural_view,
        max_per_class=20_000,
        random_state=REFERENCE_MODEL_CONFIG.random_state,
    )
reference_metrics = {
    "training": {
        "review_count": int(len(reference_train)),
        "duration_seconds": float(reference_training_seconds),
        "selected_regularization_c": float(reference_classifier.C),
        "vocabulary_size": int(len(reference_vectorizer.vocabulary_)),
    }
}
for view_name, view in reference_views.items():
    reference_metrics[view_name] = evaluate_predictions(
        view,
        bootstrap_rounds=200,
        random_state=REFERENCE_MODEL_CONFIG.random_state,
    )

# Recompute both models on exact shared IDs instead of assuming membership.
# Пересчитываем обе модели на одинаковых ID, не предполагая равенство выборок.
size_comparison_rows = []
for split_name in (
    "test_product",
    "test_temporal_seen",
    "test_conditioners",
):
    current_split = evaluation_predictions.loc[
        evaluation_predictions["split_name"].eq(split_name)
    ].sort_values("review_id")
    reference_split = reference_predictions.loc[
        reference_predictions["split_name"].eq(split_name)
    ].sort_values("review_id")
    shared_ids = set(current_split["review_id"]).intersection(
        reference_split["review_id"]
    )
    current_shared = current_split.loc[
        current_split["review_id"].isin(shared_ids)
    ].reset_index(drop=True)
    reference_shared = reference_split.loc[
        reference_split["review_id"].isin(shared_ids)
    ].reset_index(drop=True)
    assert current_shared["review_id"].equals(reference_shared["review_id"])
    assert current_shared["sentiment_label"].equals(
        reference_shared["sentiment_label"]
    )
    reference_shared_metrics = evaluate_predictions(
        reference_shared,
        bootstrap_rounds=200,
        random_state=MODEL_CONFIG.random_state,
    )
    current_shared_metrics = evaluate_predictions(
        current_shared,
        bootstrap_rounds=200,
        random_state=MODEL_CONFIG.random_state,
    )
    size_comparison_rows.append(
        {
            "split_name": split_name,
            "shared_reviews": int(len(shared_ids)),
            "reference_split_coverage": len(shared_ids) / len(reference_split),
            "one_million_split_coverage": len(shared_ids) / len(current_split),
            "300k_accuracy": reference_shared_metrics["accuracy"],
            "1m_accuracy": current_shared_metrics["accuracy"],
            "accuracy_change": (
                current_shared_metrics["accuracy"]
                - reference_shared_metrics["accuracy"]
            ),
            "300k_macro_f1": reference_shared_metrics["macro_f1"],
            "1m_macro_f1": current_shared_metrics["macro_f1"],
            "macro_f1_change": (
                current_shared_metrics["macro_f1"]
                - reference_shared_metrics["macro_f1"]
            ),
            "confidence_interval_method": "percentile_cluster_bootstrap",
        }
    )
training_size_comparison = pd.DataFrame(size_comparison_rows)
comparison_records = training_size_comparison.to_dict(orient="records")
metrics["training_size_comparison_on_shared_tests"] = comparison_records
reference_metrics["training_size_comparison_on_shared_tests"] = (
    comparison_records
)
reference_artifact_paths = save_sentiment_experiment(
    vectorizer=reference_vectorizer,
    classifier=reference_classifier,
    split_sample=reference_sample,
    evaluation_predictions=reference_predictions,
    metrics=reference_metrics,
    split_config=REFERENCE_SPLIT_CONFIG,
    model_config=REFERENCE_MODEL_CONFIG,
    dataset_version=DATASET_VERSION,
    model_directory=REFERENCE_MODEL_DIRECTORY,
    split_manifest_path=REFERENCE_SPLIT_MANIFEST_PATH,
    predictions_path=REFERENCE_PREDICTIONS_PATH,
    report_path=REFERENCE_REPORT_PATH,
)
assert REFERENCE_PREDICTIONS_PATH.is_file()
assert REFERENCE_REPORT_PATH.is_file()
display(reference_selection)
display(training_size_comparison.round(4))

## 8. Save and verify the new files

Both corrected split tables store identifiers but not review text. Each new TF-IDF model, prediction table, and report is written under its own version (`v2` reference and `1m_v3`) through same-directory temporary files where supported. Historical artifacts remain untouched.

## Сохраняем и проверяем новые файлы

Обе исправленные таблицы разбиения хранят идентификаторы без дублирования текстов. Каждая новая TF-IDF-модель, таблица предсказаний и отчёт записываются под собственной версией (`v2` reference и `1m_v3`) через временные файлы в той же папке, где это применимо. Исторические artifacts не изменяются.

In [ ]:
artifact_paths = save_sentiment_experiment(
    vectorizer=vectorizer,
    classifier=classifier,
    split_sample=sentiment_sample,
    evaluation_predictions=evaluation_predictions,
    metrics=metrics,
    split_config=SPLIT_CONFIG,
    model_config=MODEL_CONFIG,
    dataset_version=DATASET_VERSION,
    model_directory=MODEL_DIRECTORY,
    split_manifest_path=SPLIT_MANIFEST_PATH,
    predictions_path=PREDICTIONS_PATH,
    report_path=REPORT_PATH,
)

for artifact_path in artifact_paths.values():
    assert Path(artifact_path).is_file()
reloaded_classifier = joblib.load(artifact_paths["classifier"])
assert tuple(reloaded_classifier.classes_) == SENTIMENT_LABELS
assert len(pd.read_parquet(SPLIT_MANIFEST_PATH)) == len(sentiment_sample)
assert len(pd.read_parquet(PREDICTIONS_PATH)) == len(evaluation_predictions)
display(pd.Series(artifact_paths, name="path").to_frame())

## 9. Conclusion and limitations

The new 1M TF-IDF and DistilBERT experiments use the same 999,999 training IDs and 269,021 validation/final-test IDs from corrected UTC split `1m_v3`. Selecting between two TF-IDF settings took 137.87 seconds; validation selected `C=0.5` and the 100,000-feature vocabulary.

On held-out Beauty products, the 1M TF-IDF reaches 84.731% natural accuracy, 72.757% natural macro F1, and 78.717% balanced macro F1. Later 2023 reviews reach 84.722% natural accuracy and 78.435% balanced macro F1. Held-out Conditioner parent products reach 86.288% natural accuracy and 77.808% balanced macro F1. All 95% intervals use whole-product cluster bootstrap.

Against the newly retrained corrected 300k reference on shared IDs, the 1M model improves natural accuracy/macro F1 by 0.951/0.957 percentage points on held-out products, 0.910/0.999 points on 87,669 shared temporal rows, and 0.987/1.100 points on held-out Conditioner parent products. The gain is consistent but moderate.

The star-derived labels still contain possible text-rating disagreements, and Neutral remains the weakest class. TF-IDF is the accepted classical reference point, not the final seller-facing model. The next notebook compares it with DistilBERT on the now identical data.

## Вывод и ограничения

Новые эксперименты 1M TF-IDF и DistilBERT используют одинаковые 999 999 train ID и 269 021 validation/final-test ID исправленного UTC-разбиения `1m_v3`. Выбор из двух настроек TF-IDF занял 137,87 секунды; validation выбрала `C=0.5` и словарь из 100 000 признаков.

На held-out товарах Beauty 1M TF-IDF получила natural accuracy 84,731%, natural macro F1 72,757% и balanced macro F1 78,717%. Для поздних отзывов 2023 года результаты равны 84,722% и 78,435%; для held-out Conditioner parent products — 86,288% и 77,808%. Все 95%-интервалы рассчитаны cluster bootstrap целых товаров.

Относительно заново обученной исправленной 300k-reference на общих ID 1M-модель улучшает natural accuracy/macro F1 на 0,951/0,957 процентного пункта для held-out товаров, на 0,910/0,999 пункта для 87 669 общих temporal-строк и на 0,987/1,100 пункта для held-out Conditioner parent products. Улучшение стабильно, но умеренно.

В метках по звёздам всё ещё возможны несовпадения текста и оценки, а Neutral остаётся самым слабым классом. TF-IDF принимается как классическая точка сравнения, но не как готовая модель для продавца. Следующий ноутбук сравнивает её с DistilBERT уже на полностью одинаковых данных.